# Reranker Fine-tune — Conclusive Proof of Concept

**The question this notebook answers**: does fine-tuning a multilingual cross-encoder on Swiss-legal in-domain data deliver enough lift to be worth investing in a full-scale training pipeline?

**Hypothesis** (to confirm or reject): the prior off-the-shelf reranker failures were a *training-data problem*, not a *model-capacity problem*. In-domain fine-tuning of the SAME Qwen3-Reranker-8B (your prior diagnostic's best) on 30k Swiss-legal (English-question, statute-text) triplets — same model architecture, same eval — should produce a meaningful R@K lift on val.

**Experiment design**:
1. **Phase 2** — Baseline: score val's Stage-A top-2000 candidates with off-the-shelf Qwen3-Reranker-8B.
2. **Phase 3** — Train: LoRA fine-tune Qwen3-Reranker-8B on 30k self-supervised triplets (`reranker_train_30k.parquet`).
3. **Phase 4** — Fine-tuned: score the SAME val candidates with the LoRA-merged model.
4. **Phase 5** — Verdict: side-by-side R@K table + decision criteria.

**Decision gate**:
- **macro R@K_gold lift ≥ 30%** relative → full-scale fine-tune justified, proceed
- **lift 10-30%** → marginal, need more training data / better negatives
- **lift < 10%** → re-architect

**Required inputs on Drive**:
- `/MyDrive/swiss_law/research/stage_b_input/stage_b_input.parquet`  (already there)
- `/MyDrive/swiss_law/data/val.csv`  (already there)
- `/MyDrive/swiss_law/research/anchor_funnel_val001_v7/snapshot/gold_doc_sets.json`  (already there)
- `/MyDrive/swiss_law/research/reranker_train/reranker_train_30k.parquet`  (**upload this — 10.6 MB**)

## Phase 0 — Setup

In [ ]:
import os, sys, subprocess, json, time, gc, io, re, math
from pathlib import Path
sys.stdout = io.TextIOWrapper(sys.stdout.buffer, encoding="utf-8", errors="replace")

IS_COLAB = "google.colab" in sys.modules
print(f"Colab: {IS_COLAB}")
if IS_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    # vLLM for inference, peft for LoRA training, accelerate for training loop
    subprocess.run(["pip","install","-q","-U",
        "vllm>=0.9.1","transformers>=4.51.0","peft>=0.13.0","accelerate>=1.0.0",
        "datasets>=3.0.0","bitsandbytes>=0.44.0",
        "pandas==2.2.3","pyarrow==16.1.0","numpy==1.26.4","tqdm"], check=True)
print("Setup OK")


## Phase 1 — Paths, val data, Stage A candidates

In [ ]:
DRIVE_ROOT     = Path("/content/drive/MyDrive/swiss_law")
STAGE_B_IN     = DRIVE_ROOT / "research" / "stage_b_input" / "stage_b_input.parquet"
VAL_CSV        = DRIVE_ROOT / "data"     / "val.csv"
GOLD_SETS      = DRIVE_ROOT / "research" / "anchor_funnel_val001_v7" / "snapshot" / "gold_doc_sets.json"
TRAIN_PARQ     = DRIVE_ROOT / "research" / "reranker_train" / "reranker_train_30k.parquet"

OUT_DIR        = DRIVE_ROOT / "research" / "reranker_finetune_poc"
OUT_DIR.mkdir(parents=True, exist_ok=True)
LORA_DIR       = OUT_DIR / "lora_adapter"
LORA_DIR.mkdir(parents=True, exist_ok=True)

MODEL_NAME     = "Qwen/Qwen3-Reranker-8B"
TOP_K_EVAL     = 2000     # eval all of Stage A top-2000 per query (full pool)
INSTR          = ("Given a Swiss legal query in English, find the most relevant "
                  "Swiss legal text (statute or precedent in German/French/Italian).")

# Training hyperparams
LR             = 1e-4
BATCH_SIZE     = 8
GRAD_ACCUM     = 2          # effective batch 16
EPOCHS         = 1
LORA_RANK      = 16
LORA_ALPHA     = 32
MAX_PROMPT_LEN = 1024

print(f"Verifying inputs:")
for p in [STAGE_B_IN, VAL_CSV, GOLD_SETS, TRAIN_PARQ]:
    print(f"  {'OK' if p.exists() else 'MISSING'}  {p}")

import pandas as pd
val_df = pd.read_csv(VAL_CSV)
QIDS = sorted(val_df.query_id.unique().tolist())
qid_to_query = {r.query_id: str(r.query) for r in val_df.itertuples()}

gold_sets = json.load(open(GOLD_SETS, encoding="utf-8"))
qid_to_gold = {q: set(gold_sets.get(q, [])) for q in QIDS}
gold_totals = {q: len(qid_to_gold[q]) for q in QIDS}

sb_all = pd.read_parquet(STAGE_B_IN)
print(f"\nVal queries: {len(QIDS)}, Stage A rows: {len(sb_all):,}")
print(f"Gold counts: {gold_totals}")

# Build (qid, did, text) eval set: top-2000 per query
eval_pairs = []
for q in QIDS:
    sub = sb_all[sb_all.qid == q].sort_values("stage_a_rank").head(TOP_K_EVAL).reset_index(drop=True)
    for r in sub.itertuples():
        eval_pairs.append({
            "qid":    q,
            "did":    r.did,
            "is_gold":bool(r.is_gold),
            "stage_a_rank": int(r.stage_a_rank),
            "citation": r.citation,
            "text":   (r.text or "")[:1000],  # match prior diagnostic max length
        })
eval_df = pd.DataFrame(eval_pairs)
print(f"\nEval pairs total: {len(eval_df):,}  (~{TOP_K_EVAL} per query × {len(QIDS)} queries)")
print(f"Gold in eval set: {int(eval_df.is_gold.sum())}")


## Phase 2 — BASELINE: Score val candidates with off-the-shelf Qwen3-Reranker-8B

Uses the same scoring approach as your prior diagnostic: vLLM, chat prompt, score = logprob("yes") - logprob("no") at the final assistant position. The instruction is the cross-lingual phrasing (your "INSTR_CROSSLING" from the prior notebook).

In [ ]:
from vllm import LLM, SamplingParams
from transformers import AutoTokenizer

print(f"Loading {MODEL_NAME} (baseline, no LoRA) via vLLM ...")
qwen3_tok = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True, padding_side="left")

# Resolve yes/no token ids
yes_id = qwen3_tok("yes", add_special_tokens=False).input_ids[0]
no_id  = qwen3_tok("no",  add_special_tokens=False).input_ids[0]
print(f"  yes_id={yes_id}, no_id={no_id}")

# vLLM stdout shim (same as prior diagnostic)
_orig_stdout, _orig_stderr = sys.stdout, sys.stderr
sys.stdout = sys.__stdout__; sys.stderr = sys.__stderr__
try:
    qwen3_llm = LLM(
        model=MODEL_NAME, dtype="bfloat16", max_model_len=MAX_PROMPT_LEN,
        gpu_memory_utilization=0.85, enforce_eager=False, trust_remote_code=True,
    )
finally:
    sys.stdout, sys.stderr = _orig_stdout, _orig_stderr
print(f"Loaded.")

def format_prompt(query, doc, instruct=INSTR):
    return (
        "<|im_start|>system\n"
        "Judge whether the Document meets the requirements based on the Query and Instruct provided. "
        "Note that the answer can only be \"yes\" or \"no\".<|im_end|>\n"
        "<|im_start|>user\n"
        f"<Instruct>: {instruct}\n"
        f"<Query>: {query}\n"
        f"<Document>: {doc}<|im_end|>\n"
        "<|im_start|>assistant\n<think>\n\n</think>\n\n"
    )

def score_pairs(llm, prompts, batch_size=64):
    """Return list of logit-diff scores."""
    sp = SamplingParams(temperature=0.0, top_p=1.0, max_tokens=1,
                        logprobs=20)  # need top-K logprobs to find yes/no
    scores = []
    for i in range(0, len(prompts), batch_size):
        batch = prompts[i:i+batch_size]
        outs = llm.generate(batch, sp)
        for o in outs:
            # logprobs at the first generated token
            lp = o.outputs[0].logprobs[0]
            yes_lp = lp.get(yes_id)
            no_lp  = lp.get(no_id)
            yes_v = yes_lp.logprob if yes_lp is not None else -1e9
            no_v  = no_lp.logprob  if no_lp  is not None else -1e9
            scores.append(yes_v - no_v)
        if i % (batch_size * 10) == 0:
            print(f"    scored {i+len(batch):,}/{len(prompts):,}")
    return scores

# Build prompts for the full eval set
print(f"\nBuilding {len(eval_df):,} prompts ...")
prompts = [format_prompt(qid_to_query[r.qid], r.text) for r in eval_df.itertuples()]
print(f"  mean prompt len (chars): {sum(len(p) for p in prompts)//len(prompts)}")

print(f"\nBaseline scoring ...")
t0 = time.time()
baseline_scores = score_pairs(qwen3_llm, prompts, batch_size=128)
print(f"Done in {(time.time()-t0)/60:.1f} min  ({len(prompts)/max(1,time.time()-t0):.0f} pairs/sec)")
eval_df["baseline_score"] = baseline_scores

# Diagnostic: gold vs non-gold mean
gold = eval_df[eval_df.is_gold]
non = eval_df[~eval_df.is_gold]
print(f"\nBaseline gold mean score:    {gold.baseline_score.mean():+.3f}  (n={len(gold)})")
print(f"Baseline non-gold mean:        {non.baseline_score.mean():+.3f}  (n={len(non)})")
print(f"Separation:                    {gold.baseline_score.mean()-non.baseline_score.mean():+.3f}")

# Free vLLM before training
del qwen3_llm
gc.collect()
import torch
torch.cuda.empty_cache()
print(f"\nGPU mem after free: {torch.cuda.memory_allocated()/1024**3:.1f} GB")


## Phase 3 — LoRA fine-tune

LoRA on Qwen3-Reranker-8B's attention + MLP projections. ~150M trainable params (vs 8B base — keeps memory low). Loss: cross-entropy on the final yes/no token. ~30k positives + 30k negatives = 60k training examples, batch 8 × grad_accum 2 = effective 16, ~3,750 steps in 1 epoch.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, get_linear_schedule_with_warmup
from peft import LoraConfig, get_peft_model, TaskType
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

print(f"Loading {MODEL_NAME} for training (HF transformers + LoRA) ...")
t0 = time.time()
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, dtype=torch.bfloat16,
    device_map="cuda", trust_remote_code=True,
)
print(f"  loaded in {time.time()-t0:.1f}s  ({torch.cuda.memory_allocated()/1024**3:.1f} GB)")

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=LORA_RANK, lora_alpha=LORA_ALPHA, lora_dropout=0.05,
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
model.train()

# Build training dataset: each triplet -> 2 examples (pos = yes, neg = no)
print(f"\nLoading training data ...")
train_df = pd.read_parquet(TRAIN_PARQ)
print(f"  {len(train_df):,} triplets")

class RerankerSFTDataset(Dataset):
    def __init__(self, df, tok, max_len=MAX_PROMPT_LEN):
        self.examples = []
        for r in df.itertuples():
            for doc, is_pos in [(r.pos_text, True), (r.neg_text, False)]:
                self.examples.append((r.query, doc, is_pos))
        self.tok = tok
        self.max_len = max_len
    def __len__(self):
        return len(self.examples)
    def __getitem__(self, idx):
        q, doc, is_pos = self.examples[idx]
        target = "yes" if is_pos else "no"
        full = format_prompt(q, doc) + target
        enc = self.tok(full, return_tensors="pt", add_special_tokens=False,
                       max_length=self.max_len, truncation=True)
        ids = enc.input_ids[0]
        labels = torch.full_like(ids, -100)
        labels[-1] = ids[-1]   # only supervise the final yes/no token
        return {"input_ids": ids, "labels": labels}

def collate(batch):
    max_len = max(b["input_ids"].size(0) for b in batch)
    pad_id = qwen3_tok.pad_token_id or qwen3_tok.eos_token_id
    input_ids = torch.full((len(batch), max_len), pad_id, dtype=torch.long)
    labels    = torch.full((len(batch), max_len), -100, dtype=torch.long)
    attention_mask = torch.zeros((len(batch), max_len), dtype=torch.long)
    for i, b in enumerate(batch):
        L = b["input_ids"].size(0)
        # left-pad
        input_ids[i, -L:] = b["input_ids"]
        labels[i, -L:]    = b["labels"]
        attention_mask[i, -L:] = 1
    return {"input_ids": input_ids, "labels": labels, "attention_mask": attention_mask}

ds = RerankerSFTDataset(train_df, qwen3_tok)
dl = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate, num_workers=2)
print(f"  dataset: {len(ds):,} examples, dataloader: {len(dl):,} batches per epoch")

total_steps = len(dl) * EPOCHS // GRAD_ACCUM
optim = AdamW([p for p in model.parameters() if p.requires_grad], lr=LR)
sched = get_linear_schedule_with_warmup(optim, num_warmup_steps=int(0.03*total_steps),
                                        num_training_steps=total_steps)

print(f"\nTraining: {total_steps:,} optimizer steps over {EPOCHS} epoch(s) ...")
print(f"  Effective batch: {BATCH_SIZE*GRAD_ACCUM}, LR={LR}, max_len={MAX_PROMPT_LEN}")
t_start = time.time()
running_loss = 0.0
step = 0
optim.zero_grad()
for epoch in range(EPOCHS):
    for i, batch in enumerate(dl):
        batch = {k: v.cuda() for k, v in batch.items()}
        out = model(**batch)
        loss = out.loss / GRAD_ACCUM
        loss.backward()
        running_loss += out.loss.item()
        if (i + 1) % GRAD_ACCUM == 0:
            torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], 1.0)
            optim.step(); sched.step(); optim.zero_grad()
            step += 1
            if step % 50 == 0:
                avg = running_loss / (GRAD_ACCUM * 50)
                running_loss = 0.0
                elapsed = (time.time()-t_start) / 60
                eta = elapsed / step * (total_steps - step)
                print(f"  step {step:>5}/{total_steps}  loss={avg:.4f}  lr={sched.get_last_lr()[0]:.2e}  "
                      f"elapsed={elapsed:.1f}min  eta={eta:.1f}min")
print(f"\nTraining done in {(time.time()-t_start)/60:.1f} min")

# Save adapter
model.save_pretrained(str(LORA_DIR))
print(f"\nLoRA adapter saved -> {LORA_DIR}")
print(f"  contents: {list(os.listdir(LORA_DIR))}")

# Free training model
del model, optim, sched, dl, ds
gc.collect()
torch.cuda.empty_cache()
print(f"GPU mem after free: {torch.cuda.memory_allocated()/1024**3:.1f} GB")


## Phase 4 — FINE-TUNED EVAL: Score the same val candidates with the LoRA-merged model

We merge the LoRA adapter into the base model on-the-fly (or load it via vLLM's LoRA support). Then re-score the SAME 20,000 val pairs and compute R@K.

In [ ]:
import torch
from transformers import AutoModelForCausalLM
from peft import PeftModel

# Reload base + merge LoRA so vLLM can use the resulting model directly
print(f"Reloading base + merging LoRA adapter ...")
t0 = time.time()
base = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, dtype=torch.bfloat16, device_map="cpu", trust_remote_code=True
)
merged = PeftModel.from_pretrained(base, str(LORA_DIR))
merged = merged.merge_and_unload()    # collapse LoRA into base weights
MERGED_DIR = OUT_DIR / "qwen3_reranker_8b_swiss_legal_merged"
merged.save_pretrained(str(MERGED_DIR), safe_serialization=True)
qwen3_tok.save_pretrained(str(MERGED_DIR))
print(f"Merged model saved to {MERGED_DIR} in {time.time()-t0:.1f}s")

del base, merged
gc.collect()
torch.cuda.empty_cache()

# Load via vLLM
print(f"\nLoading merged model via vLLM ...")
_orig_stdout, _orig_stderr = sys.stdout, sys.stderr
sys.stdout = sys.__stdout__; sys.stderr = sys.__stderr__
try:
    qwen3_llm_ft = LLM(
        model=str(MERGED_DIR), dtype="bfloat16", max_model_len=MAX_PROMPT_LEN,
        gpu_memory_utilization=0.85, enforce_eager=False, trust_remote_code=True,
    )
finally:
    sys.stdout, sys.stderr = _orig_stdout, _orig_stderr
print(f"Loaded.")

print(f"\nFine-tuned scoring on the SAME {len(prompts):,} pairs ...")
t0 = time.time()
ft_scores = score_pairs(qwen3_llm_ft, prompts, batch_size=128)
print(f"Done in {(time.time()-t0)/60:.1f} min")
eval_df["ft_score"] = ft_scores

# Diagnostic
gold = eval_df[eval_df.is_gold]
non = eval_df[~eval_df.is_gold]
print(f"\nFine-tuned gold mean:     {gold.ft_score.mean():+.3f}  (n={len(gold)})")
print(f"Fine-tuned non-gold mean:   {non.ft_score.mean():+.3f}  (n={len(non)})")
print(f"Fine-tuned separation:      {gold.ft_score.mean()-non.ft_score.mean():+.3f}")
print(f"Baseline separation was:    {eval_df[eval_df.is_gold].baseline_score.mean()-eval_df[~eval_df.is_gold].baseline_score.mean():+.3f}")

del qwen3_llm_ft
gc.collect()
torch.cuda.empty_cache()


## Phase 5 — VERDICT: R@K side-by-side, F1 at K_gold, per-query lift

The decision criterion: macro R@K_gold lift ≥ 30% relative on val. If yes -> full-scale fine-tune is justified.

In [ ]:
def f1(p, r): return 0.0 if (p+r)==0 else 2*p*r/(p+r)

K_REPORT = [10, 25, 50, 100, 200, 500]

# Compute R@K (macro avg per-query) for both score columns
def per_query_RatK(eval_df, score_col):
    out = {K: [] for K in K_REPORT + ["gold"]}
    for q in QIDS:
        sub = eval_df[eval_df.qid == q].sort_values(score_col, ascending=False)
        gold_total = gold_totals[q]
        for K in K_REPORT:
            top = sub.head(K)
            r_at_k = int(top.is_gold.sum()) / max(1, gold_total)
            out[K].append((q, r_at_k))
        # R@K_gold (the most directly relevant for Stage B operating at K=K_gold)
        K_gold = gold_total
        top = sub.head(K_gold)
        correct = int(top.is_gold.sum())
        p = correct / max(1, K_gold); r = correct / max(1, gold_total)
        out["gold"].append((q, p, r, f1(p, r), correct))
    return out

print("="*90)
print("  RERANKER FINE-TUNE — BASELINE vs FINE-TUNED on val (Stage A top-2000)")
print("="*90)

base_rk = per_query_RatK(eval_df, "baseline_score")
ft_rk   = per_query_RatK(eval_df, "ft_score")

# Macro R@K
print(f"\n{'K':<6}{'baseline':>12}{'fine-tuned':>14}{'delta':>10}{'rel %':>10}")
for K in K_REPORT:
    base_macro = sum(v for _, v in base_rk[K]) / 10
    ft_macro   = sum(v for _, v in ft_rk[K]) / 10
    delta = ft_macro - base_macro
    rel = (delta / max(1e-6, base_macro)) * 100
    print(f"R@{K:<4}{base_macro:>12.4f}{ft_macro:>14.4f}{delta:>+10.4f}{rel:>+9.1f}%")

# F1 at K_gold per-query
print(f"\n=== F1 at K=K_gold (the Stage B operating point) ===")
print(f"{'qid':<8}{'gold':>5}{'baseline R':>14}{'baseline F1':>14}{'ft R':>10}{'ft F1':>10}{'F1 delta':>12}")
base_F1s, ft_F1s = [], []
for (q, p_b, r_b, f1_b, c_b), (_, p_f, r_f, f1_f, c_f) in zip(base_rk["gold"], ft_rk["gold"]):
    base_F1s.append(f1_b); ft_F1s.append(f1_f)
    print(f"  {q:<6}{gold_totals[q]:>5}{r_b:>14.3f}{f1_b:>14.3f}{r_f:>10.3f}{f1_f:>10.3f}{f1_f-f1_b:>+12.3f}")
macro_base = sum(base_F1s)/10; macro_ft = sum(ft_F1s)/10
print(f"\n  MACRO F1   baseline = {macro_base:.4f}     fine-tuned = {macro_ft:.4f}     delta = {macro_ft-macro_base:+.4f}")
if macro_base > 0:
    rel = (macro_ft - macro_base) / macro_base * 100
    print(f"  Relative lift on macro F1: {rel:+.1f}%")

# Macro R@K_gold (the headline number)
base_rkgold = sum(r for _, _, r, _, _ in base_rk["gold"]) / 10
ft_rkgold   = sum(r for _, _, r, _, _ in ft_rk["gold"])   / 10
rel_rkgold = (ft_rkgold - base_rkgold) / max(1e-6, base_rkgold) * 100

print(f"\n" + "="*90)
print(f"  HEADLINE METRIC: macro Recall@K_gold")
print(f"="*90)
print(f"  Baseline:    {base_rkgold:.4f}")
print(f"  Fine-tuned:  {ft_rkgold:.4f}")
print(f"  Lift:        {ft_rkgold-base_rkgold:+.4f}  ({rel_rkgold:+.1f}% relative)")

print(f"\n=== VERDICT ===")
if rel_rkgold >= 30.0:
    print(f"  CONCLUSIVE YES — relative lift {rel_rkgold:+.1f}% ≥ 30%.")
    print(f"  Full-scale fine-tune (5M-edge self-supervised + synthetic queries) is justified.")
    print(f"  Expected production F1: 0.30-0.50 macro on val (scaling lift × 5-10).")
elif rel_rkgold >= 10.0:
    print(f"  MARGINAL — relative lift {rel_rkgold:+.1f}% in [10%, 30%).")
    print(f"  Full-scale training MAY work but needs better negatives or query synthesis.")
    print(f"  Recommend: try harder negatives (top-100 retrieved non-gold per query) before committing.")
else:
    print(f"  CONCLUSIVE NO — relative lift {rel_rkgold:+.1f}% < 10%.")
    print(f"  Fine-tuning this reranker on this signal does not transfer to val.")
    print(f"  Re-architect: try fine-tuning embedding model + dual-encoder retrieval instead.")


## Phase 6 — Save outputs

In [ ]:
eval_df.to_parquet(OUT_DIR / "val_scores_baseline_vs_ft.parquet", index=False)
with open(OUT_DIR / "verdict.json", "w") as f:
    json.dump({
        "macro_R_at_K_gold_baseline": base_rkgold,
        "macro_R_at_K_gold_finetuned": ft_rkgold,
        "relative_lift_pct": rel_rkgold,
        "macro_F1_baseline": macro_base,
        "macro_F1_finetuned": macro_ft,
        "per_query_R_at_K": {K: dict(base_rk[K]) for K in K_REPORT},
        "per_query_F1_baseline": dict(zip(QIDS, base_F1s)),
        "per_query_F1_finetuned": dict(zip(QIDS, ft_F1s)),
        "training_config": {
            "model": MODEL_NAME, "rank": LORA_RANK, "alpha": LORA_ALPHA,
            "epochs": EPOCHS, "batch_size": BATCH_SIZE, "grad_accum": GRAD_ACCUM,
            "lr": LR, "max_prompt_len": MAX_PROMPT_LEN,
        },
    }, f, indent=2)
print(f"Saved to {OUT_DIR}")
